# TRPG 마스터봇 (룰북 + 시나리오)

| 파트 | 담당 | 셀 |
|---|---|---|
| 0. 공통 | 같이 | 설정값(모델 qwen3.5:9b 공통) · 설치(버전 고정) · Ollama · 데이터 받기(깃허브) |
| 1. 룰북 RAG | 민관기 | R1 ~ R8 (Ex01 ~ Ex16) |
| 2. 시나리오 RAG | 김재혁 | S1 ~ S8 (##1 ~ ##12) |
| 3. 입력 | 장우창 | I1 ~ I4 |
| 4. 출력 | 배준호 | O1 ~ O6 |

- 겹치는 이름은 룰북 쪽에 `rule`을 붙임: `rulechunks`, `rulebase`, `rulesearch`, `ruleevaluate`, `rulerank`, `rulehit`, `rulechat`, `RULE_TOP_K`
- `##9`, `##12`, `I4`는 입력을 기다리는 셀 → **모두 실행** 중에는 `##9`·`##12`는 빈 줄(Enter), `I4`는 `exit`로 넘기기


# ⬜ 0. 공통


## ⬜ 설정값

In [ ]:
# 설정값

OLLAMA_URL = "http://localhost:11434"     # Ollama 서버 주소
EMBED_MODEL = "bge-m3"                    # 임베딩 모델 (글 → 숫자 벡터)
GEN_MODEL = "qwen3.5:9b"                  # 생성 모델 (룰북 번역·선택·검증 + 시나리오 + 출력 공통)
TRANSLATE_MODEL = GEN_MODEL               # 번역도 같은 모델 사용

REPO = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/Min"
LOCAL = "/content"                               # 깃허브에서 받은 파일을 둘 곳 (colab에)

OUT_PATH = f"{LOCAL}/rules_new.jsonl"         # 룰북 청크 파일
DEV_PATH = f"{LOCAL}/golden_dev2.json"        # 개발 질문 세트 149문항
FINAL_PATH = f"{LOCAL}/golden_final.json"     # 최종 질문 세트 (깃허브에 안 올림 → 실험 때만 직접 업로드)
EMB_PATH = f"{LOCAL}/rules_new_emb.npy"       # 임베딩 저장 파일 (깃허브에서 받음, 없으면 새로 계산)

# ── 검색 ──
RULE_TOP_K = 5                            # 최종으로 가져올 청크 수
POOL_K = 5                                # 검색 하나당 후보 수 (합치기 전)
SEARCH_WEIGHTS = [1, 1, 1, 0.5, 0.5, 1]   # 검색 비중: 임베딩 3개, 키워드 2개
RRF_C = 60                                # RRF 점수 = 비중 / (RRF_C + 순위)
NUM_CTX = 8192                            # 문맥 길이

# ── 근거 고르기 ──
GEN_SEED = 42                             # 같은 질문 → 같은 결과
SELECT_CHARS = 300                        # 근거 고를 때 후보마다 보여 줄 글자 수
VERIFY_CHARS = 800                        # 근거 검증 때 보여 줄 글자 수
RULE_CHARS = 4000                         # GM에게 넘길 룰북 근거 최대 글자 수 (NUM_CTX 보호)

# ── 동시 처리 ──
NUM_PARALLEL = 4                          # 모델 하나가 동시에 처리할 요청 수 (L4 기준 4)

## 0. ⬜ 설치

In [ ]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
chromadb==1.5.9
langgraph==1.2.21
transformers>=5.3,<6
sentence-transformers>=5,<6
gradio>=6,<7


Writing /content/requirements_rb.txt


In [ ]:
# 설치
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "^(pymupdf|requests|chromadb|langgraph|transformers|sentence-transformers|gradio)=="

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
chromadb==1.5.9
gradio==6.31.0
langgraph==1.2.21
pymupdf==2.0
requests==2.32.4
sentence-transformers==5.7.0
transformers==5.22.1


## 1. ⬜ Ollama 설치 · 서버 · 모델 받기
모델은 **백그라운드로** 받아요. 다운로드가 도는 동안 다음 셀들을 진행해도 돼요.

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | OLLAMA_VERSION=0.40.16 sh     # 버전 고정 (출력 파트와 같음)
!ollama --version

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# import · 서버 켜기
import os, re, json, time, math, subprocess         # 파일·경로 / 정규식 / JSON / 시간 / 수학 / 명령 실행
from collections import Counter                     # 단어 개수 세기
os.environ["ANONYMIZED_TELEMETRY"] = "False"        # Chroma 설정 통일 (출력 파트 대화 기억과 같게, chromadb import 전에)
import numpy as np, requests, chromadb              # 숫자 계산 / 웹 요청 / 벡터 DB
from tqdm.auto import tqdm                          # 진행 막대 (실행하면 보이는 그거)
from concurrent.futures import ThreadPoolExecutor   # 여러 작업 동시에 실행
from typing import TypedDict                        # 키가 정해진 딕셔너리 타입 (LangGraph 상태용)
from langgraph.graph import StateGraph, START, END  # 흐름 그래프 / 시작점 / 끝점
from sentence_transformers import SentenceTransformer  # 시나리오 카드 임베딩 (e5-small)

def wait(url, seconds=120):         # 서버가 켜질 때까지 1초마다 확인
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass                          # 아직 안 켜졌으면 에러 → 무시하고 다시
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")


os.environ["OLLAMA_KEEP_ALIVE"] = "1h"                                    # 모델을 1시간 동안 메모리에 유지
os.environ["OLLAMA_NUM_PARALLEL"] = str(NUM_PARALLEL)                     # 요청을 동시에 처리
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)    # 서버를 백그라운드로 켜기
wait(OLLAMA_URL)

서버 준비 완료: http://localhost:11434


In [ ]:
# 모델 받기
subprocess.Popen(f'nohup sh -c "ollama pull {EMBED_MODEL} && ollama pull {GEN_MODEL}" > pull.log 2>&1 &', shell=True)
print("다운로드 시작. 진행 확인은 아래 셀")

다운로드 시작. 진행 확인은 아래 셀


In [ ]:
# 모델 대기
t0 = time.time()
ready = False

while time.time() - t0 < 600:
    out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout   # 받은 모델 목록
    if GEN_MODEL in out and EMBED_MODEL in out:
        print(f"모델 준비 완료 ({int(time.time() - t0)}초 기다림)")
        ready = True
        break
    print("아직 받는 중...", int(time.time() - t0), "초")
    time.sleep(15)

if not ready:
    print("10분이 지나도 안 받아졌어요. pull.log를 확인하세요:")
    print(open("pull.log").read()[-500:])

아직 받는 중... 0 초
아직 받는 중... 15 초
아직 받는 중... 30 초
아직 받는 중... 45 초
아직 받는 중... 60 초
모델 준비 완료 (75초 기다림)


## 2. ⬜ 데이터 받기 (깃허브)


In [ ]:
# 데이터 받기 (깃허브 → /content)
for name in ["rules_new.jsonl", "golden_dev2.json"]:          # 꼭 있어야 하는 파일
    res = requests.get(f"{REPO}/{name}", timeout=60)
    res.raise_for_status()                         # 파일 이름이나 주소가 틀리면 여기서 멈춤
    with open(f"{LOCAL}/{name}", "wb") as f:
        f.write(res.content)
    print("받음:", name, len(res.content), "bytes")

# 임베딩 파일: 있으면 받고, 없으면 Ex01-b에서 새로 계산 (1~2분)
res = requests.get(f"{REPO}/rules_new_emb.npy", timeout=60)
if res.status_code == 200:
    with open(EMB_PATH, "wb") as f:
        f.write(res.content)
    print("받음: rules_new_emb.npy", len(res.content), "bytes")
else:
    print("임베딩 파일 없음 → Ex01-b에서 새로 계산")
print(OLLAMA_URL, "| LLM(번역·생성 공통):", GEN_MODEL, "| 임베딩:", EMBED_MODEL)


받음: rules_new.jsonl 1903588 bytes
받음: golden_dev2.json 19968 bytes
받음: rules_new_emb.npy 8929408 bytes
http://localhost:11434 | LLM(번역·생성 공통): qwen3.5:9b | 임베딩: bge-m3


# 1. 🟦 룰북 RAG — 민관기


## R1. 🟦 청크 읽기 · 임베딩 · Chroma 컬렉션

In [ ]:
# Ex01-a. 청크 읽기
rulechunks = []
for line in open(OUT_PATH, encoding="utf-8"):   # jsonl = 한 줄에 청크 하나
    c = json.loads(line)                        # 글자 -> 딕셔너리
    if c["meta"]["type"] == "rule":             # 규칙 청크만 검색 대상
        rulechunks.append(c)

print("검색 대상 청크:", len(rulechunks))            # 글꼴 크기(제목)로 자르되, 1500자 이상의 글은 나눠서

검색 대상 청크: 2180


In [ ]:
# Ex01-b. 임베딩

def embed(texts, model=EMBED_MODEL):   # 글 여러 개 → 벡터 여러 개
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()                       # 서버 오류면 여기서 멈춤
    return res.json()["embeddings"]


BATCH = 32

# 저장된 임베딩이 있고 청크 수도 같으면 → 다시 계산하지 않고 불러오기
saved = np.load(EMB_PATH) if os.path.exists(EMB_PATH) else None

if saved is not None and saved.shape[0] == len(rulechunks):
    emb = saved
    print("캐시 사용:", emb.shape)
else:
    vecs = []
    for i in tqdm(range(0, len(rulechunks), BATCH)):           #    32개씩 묶어서 반복
        batch = [c["text"] for c in rulechunks[i:i + BATCH]]   #    이번 묶음의 글만 뽑기
        vecs += embed(batch)                               #    서버에 요청해서 벡터를 이어 붙이기
    emb = np.array(vecs, dtype="float32")                  #    리스트 → 숫자 배열
    np.save(EMB_PATH, emb)
    print("새로 계산:", emb.shape)

# 캐시 사용: (2180, 1024)

캐시 사용: (2180, 1024)


In [ ]:
# Ex01-c. Chroma 컬렉션
# 임베딩은 Ex01-b에서 만든 것을 그대로 넣음
client = chromadb.Client()                     # 메모리에만 만듦

# 같은 런타임에서 다시 실행해도 옛 벡터가 남지 않게, 있으면 지우고 새로 만듦
try:
    client.delete_collection("rules")
except Exception:
    pass                                       # 처음 실행이면 지울 게 없어서 에러 → 무시

rulebase = client.create_collection(
    name="rules",
    configuration={"hnsw": {"space": "cosine",    # 코사인 유사도로 비교
                            "ef_search": 200}},   # 검색할 때 더 넓게 살펴봄
)

ids = [str(i) for i in range(len(rulechunks))]                # Chroma id는 문자열: 청크 순서 번호를 그대로 사용
metadatas = [{"h1": c["meta"]["h1"]} for c in rulechunks]     # 장 이름
rulebase.add(ids=ids, embeddings=emb.tolist(), metadatas=metadatas)
print("컬렉션에 들어간 벡터 수:", rulebase.count())

# 컬렉션에 들어간 벡터 수: 2180

컬렉션에 들어간 벡터 수: 2180


## R2. 🟦 제목

In [ ]:
# Ex02. 제목 정규화
def norm(title):                                    # 제목 다듬기
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)    # "Prone [Condition]" → "Prone"
    title = re.sub(r"\s*\([^)]*\)$", "", title)     # "Hide (Action)"     → "Hide"
    title = title.replace("’", "'")                 # ’ → '
    title = re.sub(r"^Level \d+:\s*", "", title)    # "Level 5: Extra Attack" → "Extra Attack"
    return title.strip().lower()

def stem(title):
    return re.sub(r"s\b", "", norm(title))          # "attacks" → "attack"

def rulehit(title, want):                               # 정답인지 (느슨한 비교: 단수/복수 차이 무시)
    return stem(title) == stem(want)

## R4. 🟦 번역 (한국어 질문 → 영어 검색어)
- 번역①: 일반 번역 (예시 3개)
- 번역②: 답이 될 룰북 항목 이름 최대 3개


In [ ]:
# Ex03. 번역 호출

def ask(prompt, model=None, temperature=0.0):      # 번역용 LLM 호출
    options = {
        "temperature": temperature,                # 0
        "seed": GEN_SEED,
        "num_ctx": NUM_CTX,
    }
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or TRANSLATE_MODEL,       # model을 안 주면 기본 번역 모델
              "messages": [{"role": "user", "content": prompt}],
              "stream": False,
              "think": False,                          # 생각 모드 끄기 (3.5용)
              "options": options},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

In [ ]:
# Ex04. 번역 프롬프트

def clean(text):                    # 번역 결과 다듬기 (첫 줄만, "Query:"·따옴표·영어 아닌 글자 제거)
    text = text.strip().split("\n")[0]                        # 첫 줄만
    text = re.sub(r"^\s*query\s*:\s*", "", text, flags=re.I)  # 앞의 "Query:" 지우기 (대소문자 무시)
    text = re.sub(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]+", " ", text)   # 한자·일본어·한글 → 공백 (예: "Poison中毒效果" → "Poison")
    text = re.sub(r"\s+", " ", text)                          # 공백 여러 개 → 하나
    return text.strip().strip("\"'")                          # 양 끝 따옴표 지우기


# ── 번역① : 일반 번역 (예시 3개를 보여 주고 따라 하게) ──
REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
Use the official rule terms that the rulebook would use.
Keep every key part of the question: who or what it is about, and what the user wants to do or know.
Output only the query, nothing else.

Question: 숙련 보너스는 레벨마다 얼마야?
Query: Proficiency Bonus by character level

Question: 강을 헤엄쳐서 건너도 돼?
Query: Swimming rules

Question: 회피의 반지가 뭐야?
Query: Ring of Evasion

Question: {question}
Query:"""


def translate1(question):           # 번역① 일반 번역
    prompt = REWRITE_TEMPLATE.format(question=question)     # format() : 문자열 안의 {} 빈칸에 값을 끼워 넣는 함수
    return clean(ask(prompt))

# ── 번역② : 답이 될 룰북 항목 이름 최대 3개 ──
REWRITE2 = """Name the D&D 5.2 rulebook entries that could answer the user's question:
the rule, action, condition, spell, class feature, item, or creature, written as its heading in the rulebook.
Give up to 3 names on one line, separated by commas, most likely first. Output only the names, nothing else.

Question: {q}
Names:"""

def translate2(question):
    prompt = REWRITE2.format(q=question)
    return clean(ask(prompt))

## R5. 🟦 키워드 검색 BM25
임베딩은 **뜻**, BM25는 **글자 그대로의 단어**를 비교

In [ ]:
# Ex05. 키워드 검색
# 불용어제거
STOP = set("a an the of to in on and or is are be can you your it its as at by for with that this if when how what do does".split())


def tokenize(text):                 # 글 → 단어 목록 (소문자, 영어·숫자만, 불용어 제외)
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOP]


class BM25:
    # 점수가 높아지는 경우: 단어가 많이 나옴(tf) + 드문 단어(idf) + 짧은 청크(len)
# k1 : 단어가 많이 나올 때 점수가 얼마나 오를지, b : 긴 청크를 얼마나 깎을지
    def __init__(self, docs, k1=1.5, b=0.75):      # 청크들의 단어 통계 미리 계산
        self.k1 = k1
        self.b = b
        self.tf = [Counter(d) for d in docs]       # 청크마다 단어별 등장 횟수
        self.len = np.array([len(d) for d in docs], dtype="float32")   # 청크마다 단어 수
        self.avg = self.len.mean()                 # 평균 단어 수

        # df = 그 단어가 나오는 청크 수 (한 청크에서 여러 번 나와도 1번으로)
        df = Counter()
        for d in docs:
            for w in set(d):                       # set() : 중복제거
                df[w] += 1

        # idf = 드문 단어일수록 큰 값
        total = len(docs)                          # 전체 청크 수
        self.idf = {}
        for w, n in df.items():
            self.idf[w] = math.log(1 + (total - n + 0.5) / (n + 0.5))

    def scores(self, query):                       # 질문 → 청크마다 점수
        s = np.zeros(len(self.tf), dtype="float32")# 청크마다 점수 0으로 시작
        for w in set(tokenize(query)):             # 질문의 단어를 하나씩 (중복 제거- set())
            if w not in self.idf:                  # 룰북에 없는 단어는 건너뜀
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(w, 0)                   # 이 청크에서 단어가 나온 횟수
                if f:
                    scale = 1 - self.b + self.b * self.len[i] / self.avg   # 길이 보정 (긴 청크일수록 큼)
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * scale)
                    # 희귀도 × 빈도 효과 / 길이 보정
        return s


# 제목을 두 번 넣어서 제목 단어에 가중치 -> 제목이 점수를 더 받게 하려구
docs = []
for c in rulechunks:
    title = c["meta"]["title"]      # 딕셔너리 안의 딕셔너리에서 값을 꺼내는 방식
    docs.append(tokenize(title + " " + title + " " + c["text"]))
bm25 = BM25(docs)

def keyword(question, k=10):        # 키워드 검색 → 점수 높은 청크 k개
    order = np.argsort(-bm25.scores(question))            # 점수 큰 순서의 번호 (-를 붙여 내림차순)
    return [rulechunks[i] for i in order[:k]]

## R6. 🟦 최종 검색

In [ ]:
# Ex06. 제목 검색 (⑪)

heads = []
seen = set()
for c in rulechunks:
    key = (c["meta"]["title"], c["meta"]["page"])
    if key not in seen:
        seen.add(key)
        heads.append(c)

# 제목만 임베딩 (처음 한 번)
vecs = []
for i in range(0, len(heads), BATCH):
    batch = [c["meta"]["title"] for c in heads[i:i + BATCH]]
    vecs += embed(batch)
headvec = np.array(vecs, dtype="float32")
headvec = headvec / np.linalg.norm(headvec, axis=1, keepdims=True)   # 길이 1로 → 내적 = 코사인 유사도
print("제목 수:", len(heads))

def titlesearch(names, k=POOL_K):        # 번역2 이름들 → 제목이 가장 비슷한 청크 k개
    vecs = np.array(embed(names), dtype="float32")
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    sims = vecs @ headvec.T                 # 이름 × 제목 유사도 표
    best = sims.max(axis=0)                 # 제목마다 가장 잘 맞는 이름의 점수
    order = np.argsort(-best)[:k]           # 점수 높은 순 k개
    return [heads[i] for i in order]

제목 수: 1955


In [ ]:
# Ex07. 최종 검색

def semantic(texts, k=5):           # 임베딩 검색 (뜻으로 찾기: 글 여러 개 → 비슷한 청크 k개)
    vectors = embed(texts)
    res = rulebase.query(query_embeddings=vectors, n_results=k)

    # 같은 청크가 여러 번 나오면 가장 높은 점수만 남김
    best = {}
    for ids, dists in zip(res["ids"], res["distances"]):     # 글마다 결과 k개
        for key, dist in zip(ids, dists):                    # zip() : 여러 리스트를 같은 순서끼리 짝지어 한 쌍씩 꺼내 주는 함수
            i = int(key)                                     # Chroma id = 청크 순서 번호 (문자열)
            score = 1 - dist                                 # 코사인 거리 → 유사도 (클수록 비슷)
            best[i] = max(best.get(i, -1.0), score)

    order = sorted(best, key=best.get, reverse=True)   # 점수 높은 순
    return [rulechunks[i] for i in order[:k]]

def semantic2(texts, k=5):       # 임베딩 검색을 글마다 따로 (임베딩 호출은 한 번에)
    vectors = embed(texts)       # 글 여러 개를 한 번의 호출로 벡터로
    res = rulebase.query(query_embeddings=vectors, n_results=k)

    lists = []
    for ids in res["ids"]:
        found = []
        for key in ids:
            found.append(rulechunks[int(key)])
        lists.append(found)
    return lists

def merge(lists, weights, k=RULE_TOP_K, c0=RRF_C):     # 여러 검색 결과를 순위 점수(RRF)로 합치기
    # 점수 = 비중 / (c0 + 순위) → 여러 목록에서 위에 있을수록 점수가 높아짐
    score, lookup = {}, {}
    for items, w in zip(lists, weights):
        for r, c in enumerate(items, 1):           # r = 순위 (1부터)
            score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
            lookup[c["id"]] = c                    # id → 청크 (마지막에 되찾으려고)

    order = sorted(score, key=score.get, reverse=True)      # sorted() : 값들을 정렬한 새 리스트를 돌려줌
    return [lookup[i] for i in order[:k]]

pool = ThreadPoolExecutor(max_workers=NUM_PARALLEL) # 여러 작업을 동시에 실행해 주는 도구

def retrieve(question, k=RULE_TOP_K):                        # 최종 검색
    job1 = pool.submit(translate1, question)            # 번역1
    job2 = pool.submit(translate2, question)            # 번역2
    job0 = pool.submit(semantic, [question], POOL_K)    # 임베딩 : 원문 질문

    en1 = job1.result()
    en2 = job2.result()
    first = job0.result()

    pair = semantic2([en1, en2], POOL_K)             # 임베딩: 번역1, 2를 한 번에

    names = [n.strip() for n in en2.split(",") if n.strip()]   # "Grappling, Grappled" → 이름 목록
    if not names:
        names = [en2]

    lists = [
        first,                                       # 임베딩: 원문 질문
        pair[0],                                     # 임베딩: 번역1
        pair[1],                                     # 임베딩: 번역2
        keyword(en1, POOL_K),                        # 키워드: 번역1
        keyword(en2, POOL_K),                        # 키워드: 번역2
        titlesearch(names, POOL_K),                  # 제목: 번역2 이름들
    ]
    return merge(lists, SEARCH_WEIGHTS, k), [question, en1, en2]    # (청크 5개, 검색에 쓴 글 3개)

## R7. 🟦 LLM 호출 (근거 선택·검증용 짧은 답)

In [ ]:
# Ex08. LLM 호출

# 시스템 프롬프트
SYSTEM = ("You are a D&D rules assistant.")

def rulechat(prompt, model=None, system=SYSTEM, temperature=0.0, seed=GEN_SEED, limit=None):
    options = {"num_ctx": NUM_CTX, "temperature": temperature}
    if seed is not None:
        options["seed"] = seed                     # seed가 있으면 같은 결과로 고정
    if limit:
        options["num_predict"] = limit             # 출력 길이 제한 (짧은 답용)

    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": prompt}],
              "stream": False,
              "think": False,
              "options": options},
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]


def ask3(prompt, model=None, system=SYSTEM):         # 짧은 답용 호출 (번호, 예/아니오) — 최대 20토큰
    return rulechat(prompt, model, system, 0.0, GEN_SEED, limit=20)

## R8. 🟦 근거 파이프라인 (LangGraph)

```
START → search(번역①② 동시 + 검색) → choose(근거 번호) → check(검증) ┬ 통과 → accept(원본 전체 근거) → END
                                                                    └ 실패 → refuse(거절) → END
```
- check: 고른 근거를 먼저 검증 → 실패하면 나머지 4개를 동시에 검증 → 검색 순위가 높은 통과 후보
- 한국어 설명은 GM이 써요

In [ ]:
# Ex09. 근거 선택
SELECT_TEMPLATE = """다음은 D&D 규칙 질문과 후보 근거입니다. 질문에 직접 답하는 근거의 번호 하나만 고르세요.
- 질문이 묻는 규칙 자체를 설명하는 근거를 고르세요. 이름만 비슷하거나 예시로 지나가듯 언급되는 근거는 고르지 마세요.
- 숫자 하나만 출력하세요.

### 질문
{question}

### 후보 근거
{cands}

### 번호"""


def select(question, found, model=None):   # 후보 중 질문에 맞는 근거 번호 고르기 (1부터)
    # 후보마다 "[번호] 제목: 본문 앞부분 300자" 한 줄씩
    lines = []
    for n, c in enumerate(found, 1):
        preview = c["text"][:SELECT_CHARS].replace("\n", " ")   # 줄바꿈 → 공백 (한 줄로)
        lines.append(f"[{n}] {c['meta']['title']}: {preview}")
    cands = "\n".join(lines)

    out = ask3(SELECT_TEMPLATE.format(question=question, cands=cands), model)

    # 답에서 첫 숫자 꺼내기. 숫자가 없거나 범위 밖이면 1번 -> llm답이 이상하면 무시하고 1등을 믿을게
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 1
    if 1 <= n <= len(found):
        return n
    return 1

In [ ]:
# Ex10. 원본 전체 읽기
# 검색은 나뉜 조각으로, GM에게 넘길 근거는 원본 전체로

def heading(chunk):                 # 청크 머리줄 (장 > 절 > 제목)
    return chunk["text"].split("\n", 1)[0]


def content(chunk):                 # 청크 본문 (둘째 줄부터)
    return chunk["text"].split("\n", 1)[1]


# 나뉜 조각끼리 모아 두기: (제목, 쪽) → {조각 번호: 청크}
parts = {}
for c in rulechunks:
    meta = c["meta"]
    if "part" not in meta:                      # 나뉘지 않은 청크는 건너뜀
        continue
    key = (meta["title"], meta["page"])
    if key not in parts:
        parts[key] = {}
    parts[key][meta["part"]] = c


def expand(chunk):                  # 나뉜 조각이면 원본 전체(모든 조각)를 이어서 돌려줌
    meta = chunk["meta"]
    if "part" not in meta:                      # 나뉘지 않은 청크는 그대로
        return chunk

    group = parts[(meta["title"], meta["page"])]

    texts = []
    for k in range(1, meta["parts"] + 1):       # 1번 조각부터 마지막 조각까지 순서대로(k = i(번호), x(문자열, 청크 등))
        texts.append(content(group[k]))
    whole = " ".join(texts)

    return {"id": chunk["id"], "text": heading(chunk) + "\n" + whole, "meta": meta}

In [ ]:
# Ex11. 근거 검증
VERIFY_TEMPLATE = """다음 D&D 규칙 질문과 근거입니다. 이 근거를 읽고 질문에 답할 수 있으면 '예', 다른 규칙을 설명하고 있어서 답할 수 없으면 '아니오'라고만 답하세요.
질문은 한국어, 근거는 영어예요. 질문에 쓴 말과 근거의 이름이 달라도 같은 규칙이면 '예'예요.
근거가 다른 규칙을 설명하면서 질문의 규칙을 지나가듯 언급만 하면 '아니오'예요.

### 질문
{question}
(영어 검색어: {translation})

### 근거
{title}: {text}

### 답 (예/아니오)"""

def verify(question, chunk, translation, limit=VERIFY_CHARS):                 # 근거가 질문에 맞는지 LLM에게 묻기 (예 → True)
    prompt = VERIFY_TEMPLATE.format(question=question, translation=translation,
                                    title=chunk["meta"]["title"],
                                    text=chunk["text"][:limit])
    out = ask3(prompt)
    out = re.sub(r"[\s\*'\"]", "", out)               # 공백·별표·따옴표 지우기 ("**예**" → "예")
    return out.startswith(("예", "네", "yes", "Yes"))

def verify2(question, chunk, texts, limit=VERIFY_CHARS):    # 근거 통과 여부 (제목이 맞으면 바로 통과, 아니면 LLM 검증)
    return match(texts, chunk) or verify(question, chunk, texts[1], limit)

def match(texts, chunk):            # 근거 제목이 번역① 안에 그대로 있는지
    t = norm(chunk["meta"]["title"])
    if re.search(r"\b" + re.escape(t) + r"\b", texts[1].lower()):
        return True
    return False

def place(chunk):
    return (chunk["meta"]["title"], chunk["meta"]["page"])

def pick(question, found, texts, n):    # 고른 근거부터 검증해서 처음 통과한 번호 (모두 실패면 None)
    seen = set()                        # 이미 검증한 규칙
    # 1단계 : 고른 번호가 범위 안이면 그 규칙(원본 전체)부터 검증
    if n is not None and 1 <= n <= len(found):
        seen.add(place(found[n-1]))
        if verify2(question, expand(found[n - 1]), texts, RULE_CHARS):
            return n

    # 2단계 : 아직 안 본 규칙만 모아서 동시에 검증 (같은 규칙의 다른 조각은 건너뜀)
    others = []
    for i in range(1, len(found) + 1):
        p = place(found[i - 1])
        if p in seen:
            continue
        seen.add(p)
        others.append(i)

    jobs = []
    for i in others:
        jobs.append(pool.submit(verify, question, expand(found[i - 1]), texts[1], RULE_CHARS))

    for i, job in zip(others, jobs):        # 순위가 높은 후보부터 → 처음 통과한 번호
        if job.result():
            return i
    return None

In [ ]:
# Ex12. 최종 함수 (LangGraph)
# 질문 → 검색 → 근거 선택 → 검증 → 근거 or 거절

class State(TypedDict, total=False):    # 노드끼리 주고받는 값
    question: str                       # 질문
    found: list                         # 검색 후보 5개
    texts: list                         # [원문, 번역1, 번역2]
    n: int                              # 고른 번호 → 검증 후 통과 (모두 실패면 None)
    result: dict                        # 최종 결과

def rulesearch(state):      # 노드: 룰북 검색
    found, texts = retrieve(state["question"])
    return {"found": found, "texts": texts}

def choose(state):          # 노드: 후보 중 근거 번호 고르기
    n = select(state["question"], state["found"])
    return {"n": n}

def check(state):           # 노드: 고른 근거부터 검증 (실패하면 나머지 동시 검증)
    n = pick(state["question"], state["found"], state["texts"], state["n"])
    return {"n": n}

def route(state):           # 갈림길: 통과 → accept, 모두 실패 → refuse
    if state["n"] is None:
        return "refuse"
    return "accept"

def accept(state):          # 노드: 검증된 근거를 GM에게 넘길 모양으로
    c = expand(state["found"][state["n"] - 1])     # 나뉜 청크면 원본 전체
    result = {
        "status": "ok",
        "title": c["meta"]["title"],
        "page": c["meta"]["page"],
        "path": heading(c),                        # 장 > 절 > 제목 <-어디?
        "text": content(c),                        # 영어 본문 (GM 프롬프트에 넣을 근거)
        "picked": state["n"],                      # 후보 중 몇 번째였는지
        "queries": state["texts"][1:],             # 번역1, 2
    }
    return {"result": result}

def refuse(state):
    candidates = []
    for x in state["found"][:3]:
        candidates.append((x["meta"]["title"], x["meta"]["page"]))
    result = {
        "status": "refuse",
        "candidates": candidates,
        "queries": state["texts"][1:],
    }
    return {"result": result}

# 그래프 만들기 노드등록 -> 연결 -> 컴파일
graph = StateGraph(State)
graph.add_node("search", rulesearch)   # 노드 이름은 "search" 그대로
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_node("accept", accept)
graph.add_node("refuse", refuse)

graph.add_edge(START, "search")
graph.add_edge("search", "choose")
graph.add_edge("choose", "check")
graph.add_conditional_edges("check", route, ["accept", "refuse"])   # 검증 결과에 따라 갈림
graph.add_edge("accept", END)
graph.add_edge("refuse", END)

app = graph.compile()

def evidence(question):         # 질문에 답하는 룰북 근거 찾기 (GM이 부르는 함수)
    state = app.invoke({"question": question})
    return state["result"]

# 2. 🟥 시나리오 RAG — 김재혁


<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/Kim/sce_rag_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🟥 시나리오 RAG (잠긴 종) — v5 통합용

시나리오 카드(`cards.json`)를 검색해서, 지금 플레이어가 볼 수 있는 카드만 가져오는 RAG.


| 목차 | 하는 일 | 셀 |
|---|---|---|
| S1 | 카드 읽기, 공개/GM 전용 나누기 | ##1~##2 |
| S2 | 현재 상태에서 검색 가능한 카드만 거르는 규칙 | ##3 |
| S3 | 카드 → 검색용 청크 | ##4 |
| S4 | 임베딩 + Chroma 저장 | ##5 |
| S5 | 상태 필터 검색 | ##6 |
| S6 | 골든셋 평가 (Hit@1, Hit@3, MRR) | ##7~##8 |
| S7 | 자유 입력으로 검색 결과만 확인 (모델 없음) | ##9 |
| S8 | 모델 준비 → 서술 규칙 프롬프트 → 자유 입력으로 모델 답 확인 | ##10~##12 |


## 🟥 S1. 시나리오 카드 정리 (##1~##2)

카드 파일을 읽고, 검색 대상(플레이어에게 보여 줄 수 있는 카드)과 GM 전용 카드를 나눕니다.

비밀이 검색에 섞이지 않게 하는 것이 목적.

In [ ]:
## 1. cards.json 가져오기
# 역할: 시나리오 카드 166장과 방 이름표를 읽는다.
CARDS_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/BAE/src/game/data/cards.json"   # [결합] 게임 패키지 카드(출력 파트와 같은 파일)로 통일
response = requests.get(CARDS_URL, timeout=60)
response.raise_for_status()
with open("cards.json", "wb") as f:
    f.write(response.content)

with open("cards.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cards = data["cards"]      # 카드 166장
rooms = data["rooms"]      # 방 id -> 한글 이름표 (27개)

In [ ]:
## 2. 공개용 / GM 전용 나누기
# 역할: player_text가 있는 카드만 검색 대상(search_cards)으로 삼는다. GM 전용 카드(player_text가 빈 카드)는 제외해 비밀이 섞이지 않게 한다.
search_cards = [c for c in cards if c["player_text"]]

kinds = Counter(c["kind"] for c in search_cards)
for banned in ["secret", "check", "monster"]:      # 비밀 종류 카드가 섞였는지 확인
    assert banned not in kinds, banned

## 🟥 S2. 검색 규칙 정하기 (##3)

"지금 이 플레이어에게 보여도 되는 카드인가?"를 판단하는 함수.

의미가 비슷해도 아직 공개 전이거나, 가 본 적 없는 방이거나, 다른 직업 전용이면 검색에서 제외.

In [ ]:
## 3. 현재 상황에서 검색 가능한 카드만 거르기
# 역할: "지금 이 플레이어에게 보여도 되는 카드인가?" (장면, 플래그, 가 본 방, 직업)

def can_search(card, state): # 필터
    if state["scene"] < card["reveal_after"]:
        return False        # 아직 공개 장면이 아님
    if card["req_flag"] and card["req_flag"] not in state["flags"]:
        return False        # 필요한 플래그가 없음 (req_flag가 빈 글자면 이 검사는 건너뜀)
    if card["room"] and card["room"] not in state["visited"]:
        return False        # 아직 가 보지 않은 방
    if card["owner"] != "all" and card["owner"] != state["cls"]:
        return False        # 다른 직업 전용 카드
    return True

## 🟥 S3. 시나리오 청크 만들기 (##4)

카드 1장 = 청크 1개

In [ ]:
## 4. 청크 만들기
# 역할: 카드 1장 = 청크 1개. "[종류] 방이름" 머리말 + player_text (GM 문장은 넣지 않는다).

card_by_id = {c["id"]: c for c in cards}      # id로 카드 원본을 바로 찾기 위한 사전

def make_chunk_text(card):
    room_name = rooms.get(card["room"], "")   # 엔딩 카드 9장은 room이 비어 있어서 .get 사용
    header = f"[{card['label']}]"
    if room_name:
        header += f" {room_name}"
    return header + "\n" + card["player_text"]

chunks = [{"id": c["id"], "text": make_chunk_text(c)} for c in search_cards]

for c, ch in zip(search_cards, chunks):       # GM 문장이 청크에 섞이지 않았는지 확인
    if c["gm_only_text"]:
        assert c["gm_only_text"] not in ch["text"], c["id"]

## 🟥 S4. 임베딩과 Chroma 저장 (##5)

청크 글을 벡터(숫자 384개)로 바꿔서 Chroma에 저장.

임베딩 모델: `intfloat/multilingual-e5-small` (가벼운 기본 모델)

In [ ]:
## 5. 임베딩 + Chroma 저장
# 역할: 청크를 벡터로 바꿔 Chroma(메모리)에 저장한다. 검색은 이 벡터로 한다.

model = SentenceTransformer("intfloat/multilingual-e5-small") # 가벼운 기본모델

texts = ["passage: " + ch["text"] for ch in chunks]
vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

client = chromadb.Client()      # 메모리에만 저장 (런타임을 끄면 사라짐)
collection = client.get_or_create_collection(
    name="scenario_cards",
    metadata={"hnsw:space": "cosine"},
)
collection.upsert(
    ids=[ch["id"] for ch in chunks],
    documents=[ch["text"] for ch in chunks],
    embeddings=vectors.tolist(),
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

## 🟥 S5. 시나리오 검색 + 상태 필터 (##6)

Chroma에서 유사도 순으로 전부 가져온 뒤

`can_search`를 통과한 카드만 남기고 앞에서 k개를 돌려줍니다.

In [ ]:
## 6. 상태를 반영한 검색 함수
# 역할: 질문과 비슷한 카드를 찾되, ① 지금 볼 수 있는 카드만(can_search) ② [v5] 지금 서 있는 방의 카드만 남긴다.

ROOM_ONLY = True    # [v5] True: 현재 방(state["room"])의 카드와 방 없는 카드(엔딩)만 검색 / False: 가 본 방 전체

def search(question, state, k=3, use_filter=True):
    q_vec = model.encode("query: " + question, normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_vec.tolist()], n_results=len(chunks))
        # 질문을 벡터로 바꿔 카드 전체를 유사한 순서로 가져옴

    room_now = state.get("room")                  # [v5] 지금 서 있는 방 (없으면 None -> 방 제한을 걸지 않음)
    # 비슷한 순서대로 훑으면서, 지금 볼 수 있는 카드만 남겨 앞에서 k개를 돌려준다
    hits = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        card = card_by_id[cid]
        if use_filter and not can_search(card, state):
            continue                              # 지금 볼 수 없는 카드는 건너뜀
        if use_filter and ROOM_ONLY and room_now and card["room"] and card["room"] != room_now:
            continue                              # [v5] 다른 방의 카드는 건너뜀 (방이 없는 엔딩 카드는 통과)
        hits.append((cid, round(1 - dist, 3)))    # (카드 id, 유사도)
    return hits[:k]

# 3. 🟨 입력 — 장우창


## 🟨 I1. 입력 분류


In [ ]:
# 사용자 입력 의도 분류 함수
def classify_intent(query):
    text = query.strip().lower()

    # 규칙 설명 요청 표현
    rule_question_patterns = [
        r"규칙",
        r"판정.*어떻게",
        r"주사위.*어떻게",
        r"어떻게.*판정",
        r"공격.*방법",
        r"피해.*계산",
        r"몇.*굴려",
    ]

    # 실제 전투 행동 표현
    combat_patterns = [
        r"공격한다",
        r"공격하겠다",
        r"때린다",
        r"휘두른다",
        r"사격한다",
        r"주문을 시전",
    ]

    # NPC 와의 대화 표현
    dialogue_patterns = [
        r"말을 건다",
        r"물어본다",
        r"묻는다",
        r"대화한다",
        r"설득한다",
    ]

    # 기술 또는 판정이 필요한 행동 표현
    skill_patterns = [
        r"은신",
        r"몰래",
        r"자물쇠.*딴다",
        r"함정.*해제",
    ]

    # 패턴 목록 중 하나라도 입력 문장과 일치하는지 확인하는 함수 (중첩 함수)
    def matches(patterns):
      # any() : 하나라도 일치하면 True
      # re.search(패턴, 검색할 문자열) : 문자열에서 정규표현식 패턴과 일치하는 부분 검색
        return any(re.search(p, text) for p in patterns)

    if matches(dialogue_patterns):
        return "world_action"

    if matches(rule_question_patterns):
        return "rule_question"

    if matches(combat_patterns):
        return "combat_action"

    if matches(skill_patterns):
        return "mixed_action"

    return "scenario_question"


## RAG 라우팅 변환 코드 (classify_intent() 와 rag_bridge.py)

# 사용자 의도와 RAG 검색 경로 연결
INTENT_TO_RAG = {
    "rule_question": "rule",
    "scenario_question": "scenario",
    "combat_action": "both",
    "world_action": "scenario",
    "mixed_action": "both"
}


# 실제 검색할 RAG 결정 함수
def route_query(query):

    if not isinstance(query, str):
        raise TypeError("입력은 문자열이어야 합니다.")

    query = query.strip()

    if not query:
        raise ValueError("입력이 비어 있습니다.")

    # 사용자 의도 분석
    user_intent = classify_intent(query)

    # RAG 경로 선택
    rag_intent = INTENT_TO_RAG[user_intent]

    # 결과 (딕셔너리) 생성
    return {
        "player_input": query,
        "user_intent": user_intent,
        "intent": rag_intent,
        "use_rule": rag_intent in ("rule", "both"),
        "use_scenario": rag_intent in ("scenario", "both")
    }

## 🟨 I2. 두 RAG 통합 검색


In [ ]:
## RAG 검색 함수 파일 작성
# [결합] 모듈 import 대신 노트북에 있는 함수 사용 (route_query: I1, retrieve: R6, search: S5)


def search_rag(query, state, k=3):

    # 입력 분류
    route = route_query(query)

    intent = route["intent"]

    # 결과 구조
    result = {
        "player_input": route["player_input"],
        "user_intent": route["user_intent"],
        "intent": intent,
        "rule_results": [],
        "scenario_results": [],
        "translated_queries": []
    }

    # RULE RAG 검색
    if route["use_rule"]:

        e = evidence(route["player_input"])   # 입력 값에 대해 규칙 RAG 검색 함수 호출

        result["translated_queries"] = e.get("queries", [])   # 번역된 검색 질의 저장

        if e["status"] == "ok":   # 검증된 규칙 - 리스트로 저장
            result["rule_results"].append({
                "source": "rule",
                "text": e["text"],
                "score": None,
                "metadata": {
                    "title": e["title"],
                    "page": e["page"],
                    "path": e["path"]
                }
            })
        # 검증 실패 처리
        elif e["status"] == "refuse":
            print("RULE RAG: 검증된 규칙 근거 없음")

    # SCENARIO RAG 검색
    if route["use_scenario"]:

        scenario_hits = search(
            question=route["player_input"],
            state=state,
            k=k,
            use_filter=True
        )

        for card_id, score in scenario_hits:

            result["scenario_results"].append({
                "source": "scenario",
                "text": card_by_id[card_id]["player_text"],   # [결합] sce_rag.card_text 대신
                "score": score,
                "metadata": {
                    "card_id": card_id
                }
            })

    return result


# 테스트하려는 게임 상태
game_state = {
    "scene": 1,
    "cls": "fighter",
    "flags": set(),
    "visited": {"village_square"}
}

# 입력 테스
test_queries = [
    "명중 판정은 어떻게 하나요?",
    "세라에게 실종자에 대해 묻는다",
    "고블린을 공격한다",
    "몰래 자물쇠를 딴다",
    "주변을 둘러본다"
]

for query in test_queries:
    result = route_query(query)

    print("입력:", query)
    print("사용자 의도:", result["user_intent"])
    print("검색 RAG:", result["intent"])
    print("-" * 40)

입력: 명중 판정은 어떻게 하나요?
사용자 의도: rule_question
검색 RAG: rule
----------------------------------------
입력: 세라에게 실종자에 대해 묻는다
사용자 의도: world_action
검색 RAG: scenario
----------------------------------------
입력: 고블린을 공격한다
사용자 의도: combat_action
검색 RAG: both
----------------------------------------
입력: 몰래 자물쇠를 딴다
사용자 의도: mixed_action
검색 RAG: both
----------------------------------------
입력: 주변을 둘러본다
사용자 의도: scenario_question
검색 RAG: scenario
----------------------------------------


## 🟨 I3. LLM 전달 데이터


In [ ]:
## JSON 형식의 파일 작성
def build_llm_handoff(search_result, game_state):

    # 게임 상태를 JSON에 저장 가능한 형태로 변환
    clean_state = {
        "scene": game_state["scene"],
        "cls": game_state["cls"],
        "flags": sorted(game_state["flags"]),
        "visited": sorted(game_state["visited"])
    }

    # RULE 검색 결과 정리
    rule_context = []

    for hit in search_result["rule_results"]:

        meta = hit.get("metadata", {})

        rule_context.append({
            "source": "rule",
            "title": meta.get("title"),
            "page": meta.get("page"),
            "text": hit["text"]
        })

    # SCENARIO 검색 결과 정리
    scenario_context = []

    for hit in search_result["scenario_results"]:

        scenario_context.append({
            "source": "scenario",
            "card_id": hit["metadata"]["card_id"],
            "score": hit["score"],
            "text": hit["text"]
        })

    # LLM 전달 데이터 생성
    payload = {
        "player_input": search_result["player_input"],
        "user_intent": search_result.get("user_intent"),
        "intent": search_result["intent"],
        "game_state": clean_state,
        "retrieved_context": {
            "rules": rule_context,
            "scenarios": scenario_context
        },
        "instructions": [
            "플레이어의 입력과 현재 게임 상태를 고려한다.",
            "규칙과 관련된 설명은 RULE 검색 결과를 근거로 한다.",
            "시나리오의 사건과 인물 정보는 SCENARIO 검색 결과를 근거로 한다.",
            "검색 결과에 없는 사실을 확정적으로 만들어내지 않는다.",
            "검색 결과가 부족하면 임의로 규칙이나 사건을 확정하지 않는다.",
            "플레이어에게 공개 가능한 정보만 응답에 사용한다."
        ]
    }

    return payload


## JSON 형식의 파일 테스트
# 기존 통합 검색 함수 호출
search_result = search_rag(
    query="세라에게 실종자에 대해 묻는다",
    state=game_state,
    k=3
)

# LLM 전달용 데이터 생성
payload = build_llm_handoff(
    search_result=search_result,
    game_state=game_state
)

print(json.dumps(
    payload,
    ensure_ascii=False,
    indent=2
))

{
  "player_input": "세라에게 실종자에 대해 묻는다",
  "user_intent": "world_action",
  "intent": "scenario",
  "game_state": {
    "scene": 1,
    "cls": "fighter",
    "flags": [],
    "visited": [
      "village_square"
    ]
  },
  "retrieved_context": {
    "rules": [],
    "scenarios": [
      {
        "source": "scenario",
        "card_id": "scn-g-rumors",
        "score": 0.868,
        "text": "마을 사람들의 소문. 대장간 노인은 \"종소리가 날마다 가까워진다\"고 하고, 여관 주인은 밤에는 우물 곁을 지나지 말라고 이른다. 실종자 가족을 만나려면 세라를 찾아가라는 말이 어디서든 나온다. 소문의 절반은 과장이지만, 종소리가 우물 쪽에서만 난다는 말은 모두 같다."
      },
      {
        "source": "scenario",
        "card_id": "scn-q1-npc-sera",
        "score": 0.868,
        "text": "세라 — 채석공 토브의 아내이며 리나의 어머니다. 세라의 나이는 30대 후반이다. 남편과 딸이 실종되어 행방과 생사를 알고 싶어 한다. 가족이 사라진 날 우물 쪽에서 종소리가 유난히 컸다고 기억한다. 리나가 들고 다니던 인형을 수색에 맡길 수 있다."
      },
      {
        "source": "scenario",
        "card_id": "scn-s1-clue-missing",
        "score": 0.851,
        "text": "실종자 셋. 모두 지난 두 달 사이, 모두 밤에, 모두 우물 근처에서 마지막으로 목격됐다. 토브(

## 🟨 I4. 입력 → 전달 데이터 확인 (exit 입력하면 끝)


In [ ]:
## 모든 파일들을 실행하는 슈퍼바이저 파일 작성
import sys
import json
from pathlib import Path

# [결합] 경로 설정 · 모듈 import · RAG 초기화(load)는 노트북 위쪽 셀에서 이미 함

# 5. 게임 상태
def create_game_state():

    return {
        "scene": 1,
        "cls": "fighter",
        "flags": set(),
        "visited": {"village_square"}
    }


# 6. 검색 실행
def run_game():


    game_state = create_game_state()

    print("\n===== TRPG Dual RAG 테스트 =====")
    print("종료하려면 exit를 입력하세요.")

    while True:

        query = input("\n플레이어 행동: ").strip()

        if query.lower() == "exit":
            print("테스트 종료")
            break

        if not query:
            print("행동을 입력해주세요.")
            continue

        try:
            result = search_rag(
                query=query,
                state=game_state,
                k=3
            )
            payload = build_llm_handoff(
            search_result=result,
            game_state=game_state
            )

            print("\n===== LLM 전달 데이터 =====")

            print(json.dumps(
                payload,
                ensure_ascii=False,
                indent=2
            ))

        except Exception as e:
            print("검색 중 오류:", e)


# run_game()      # [결합] %run main.py 대신 (끝낼 때 exit 입력)


# 4. 🟪 출력 — 배준호

<small>마지막 수정: 2026-11-27 15:58 (한국시간)</small>


## 🟪 출력 흐름

팀 입력·RAG → 대화·프롬프트 → 답변·실행 결과 → 검사·선택지 → 게임 화면

GitHub의 `BAE` 폴더를 갱신한 뒤, 통합 v1.3의 출력 O1~O6만 교체합니다.


## 🟪 O1. 파일·모델 설정

GitHub의 BAE 보조 파일을 확인하고 공통 모델 설정을 연결합니다. 같은 파일은 다시 받지 않습니다.


In [ ]:
# 마지막 수정: 2026-11-27 15:58 (한국시간) · output-preparation-20261127
"""팀 입력 JSON·원본 RAG 결과를 Ollama의 한국어 대사와 장면으로 연결합니다.

출력 파트는 검색기를 다시 만들거나 인물 지식을 추가하지 않습니다.
게임 상태 변경은 엔진이 담당하고 여기서는 전달된 결과를 표현합니다.
"""

# GitHub의 보조 코드만 연결합니다. 대화 생성의 핵심은 이 셀 아래와 O2~O5에 있습니다.
import hashlib as _bae_hashlib
import importlib.util as _bae_importlib
from pathlib import Path as _BaePath
import sys as _bae_sys
from urllib.request import urlopen as _bae_urlopen

BAE_TEAM_BINDINGS = {k: globals()[k] for k in ("json", "re", "time") if k in globals()}
BAE_FILES_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/BAE/src/"
BAE_SETUP_SHA256 = "07cb9644148c80b98a9daf4fdbb96fe5200b5918021607de03a28f4fc4843b4e"
BAE_MANIFEST_SHA256 = "ea40867d50e456283241adf4b71e5c14e5022e73746f8a7a7ff8443667c903a0"
_bae_base = _BaePath("/content") if "google.colab" in _bae_sys.modules else _BaePath.cwd()
_bae_cache = _bae_base / ".bae_cache" / BAE_SETUP_SHA256 / "runtime_setup.py"
_bae_candidates = (_bae_base / "BAE/src/runtime_setup.py", _bae_base / "src/runtime_setup.py", _bae_cache)
_bae_setup = next((p for p in _bae_candidates if p.is_file() and _bae_hashlib.sha256(p.read_bytes()).hexdigest() == BAE_SETUP_SHA256), None)
if _bae_setup is None:
    with _bae_urlopen(BAE_FILES_URL + "runtime_setup.py", timeout=60) as _bae_response:
        _bae_raw = _bae_response.read()
    if _bae_hashlib.sha256(_bae_raw).hexdigest() != BAE_SETUP_SHA256:
        raise ValueError("GitHub의 BAE 파일 버전이 다릅니다. 같은 버전의 BAE 폴더를 먼저 업로드하세요.")
    _bae_cache.parent.mkdir(parents=True, exist_ok=True)
    _bae_cache.write_bytes(_bae_raw)
    _bae_setup = _bae_cache
_bae_spec = _bae_importlib.spec_from_file_location("_bae_runtime_setup_" + BAE_SETUP_SHA256[:12], _bae_setup)
_BAE_SETUP_MODULE = _bae_importlib.module_from_spec(_bae_spec)
_bae_spec.loader.exec_module(_BAE_SETUP_MODULE)
BAE_RUNTIME_PATH, BAE_RUNTIME_PREPARE_INFO = _BAE_SETUP_MODULE.prepare_runtime(
    BAE_FILES_URL, BAE_MANIFEST_SHA256, target=globals().get("_BAE_HELPER_TEST_PATH"),
    previous_root=globals().get("BAE_RUNTIME_PATH"))


def _bae_load_output_support(name):
    """반복 처리 보조 함수도 원래 출력 셀과 같은 공간에서 정의합니다."""
    _BAE_SETUP_MODULE.load_support(globals(), name.removesuffix(".py"))


# 이전 출력 버전의 파일 준비 함수가 진단 기록에 섞이지 않게 정리합니다.
for _bae_old_name in ("_bae_prepare_direct_helpers", "_bae_capture_team_bindings"):
    globals().pop(_bae_old_name, None)

# O1. 공통 모델 설정을 출력에 연결하고 실행 제어 도구를 준비합니다.
# 이 셀 앞부분의 GitHub 파일 준비는 최종 배포 단계에서 붙입니다.
import json
import hashlib
import re
import secrets
import threading
import time
from contextlib import contextmanager
from contextvars import ContextVar
from copy import deepcopy
_SOURCE_SHA256 = None
OUTPUT_VERSION = "output-preparation-20261127"
MODEL_NAME = GEN_MODEL
OLLAMA_BASE_URL = OLLAMA_URL
OLLAMA_KEEP_ALIVE = "30m"
OLLAMA_NUM_CTX = NUM_CTX
MAX_NEW_TOKENS = 384          # 한 번의 본문 생성 한도. 잘린 응답은 그대로 표시하지 않습니다.
MAX_INPUT_CHARACTERS = 24000  # 요청이 지나치게 커지면 오래된 대화부터 줄입니다.
HISTORY_MESSAGES = 12         # 현재 NPC와 나눈 최근 발언을 가져오는 최대 개수입니다.
SUGGESTION_COUNT = 2          # 생성 추천의 기본 개수. 실제 호출에서 조정할 수 있습니다.
DO_SAMPLE = False            # 임의 추출을 끄고 같은 설정의 답변 변동을 줄입니다.
TEMPERATURE, TOP_P, TOP_K = .7, .8, 20
output_model, input_namespace = None, None
_OUTPUT_RUNTIME_INFO = {"runtime_package": deepcopy(globals().get("BAE_RUNTIME_PREPARE_INFO", {}))}     # 실제 GPU·모델 연결 상태를 복사 기록에 함께 남깁니다.
generation_lock = threading.RLock()  # 동시에 여러 생성 요청이 GPU를 점유하지 않게 합니다.
_output_stream_sink = ContextVar("trpg_output_stream_sink", default=None)  # 현재 요청의 화면 콜백
_output_stream_cancel_events = ContextVar("trpg_output_stream_cancel_events", default=())

_bae_load_output_support("execution_control.py")


## 🟪 O2. 팀 입력 JSON 연결

팀 I2·I3를 그대로 호출해 질문·게임 상태·검색 근거를 전달받습니다.


In [ ]:
# O2. 팀 I2 검색과 I3 JSON을 그대로 연결합니다.
# 실제 발언과 검색용 문장을 구분하고, 원본 검색 결과를 출력 자료로 받습니다.
_bae_load_output_support("input_helpers.py")

def bind_team_backend(search_rag, handoff_builder, scenario_cards=None, can_search=None,
                      *, settings=None, gen_model=None, ollama_url=None, num_ctx=None):
    """준비된 팀 I2·I3 함수와 공통 설정을 연결합니다. 여기서는 검색·모델 준비를 실행하지 않습니다.

    settings=globals()로 전달하면 GEN_MODEL·OLLAMA_URL·NUM_CTX만 읽습니다.
    scenario_cards와 can_search는 UI 연결용 원본 참조이며 지식 보충에 쓰지 않습니다.
    """
    global input_namespace, MODEL_NAME, OLLAMA_BASE_URL, OLLAMA_NUM_CTX
    if not callable(search_rag) or not callable(handoff_builder):
        raise TypeError("팀 검색 함수와 handoff 생성 함수가 필요합니다.")
    if can_search is not None and not callable(can_search):
        raise TypeError("can_search는 원본 공개 조건 함수여야 합니다.")
    config = settings or {}
    model = str(gen_model or config.get("GEN_MODEL", MODEL_NAME)).strip()
    url = str(ollama_url or config.get("OLLAMA_URL", OLLAMA_BASE_URL)).rstrip("/")
    context = num_ctx if num_ctx is not None else config.get("NUM_CTX", OLLAMA_NUM_CTX)
    if not model or not url.startswith(("http://", "https://")) or type(context) is not int or context <= MAX_NEW_TOKENS:
        raise ValueError("팀의 모델 이름·Ollama 주소·문맥 한도를 확인하세요.")
    with generation_lock:
        if output_model is not None and (getattr(output_model, "model", None) != model
                or str(getattr(output_model, "base_url", "")).rstrip("/") != url
                or getattr(output_model, "num_ctx", None) != context):
            raise ValueError("이미 준비한 출력 연결과 팀 설정이 다릅니다. 새 런타임에서 연결하세요.")
        MODEL_NAME, OLLAMA_BASE_URL, OLLAMA_NUM_CTX = model, url, context
        input_namespace = {"search_rag": search_rag, "handoff_builder": handoff_builder,
            "scenario_cards": scenario_cards, "can_search": can_search,
            "memory_embedding_model": None, "input_version": "team"}
    return get_handoff


def get_handoff(player_input, state, npc_name="", history=None):
    """대화 대상이 있는 검색 질의와 실제 플레이어 발언을 분리합니다.

    검색은 팀 I2·I3를 한 번씩 그대로 사용합니다. 짧은 질문도 현재 누구에게
    한 말인지 검색기가 알 수 있게 이름만 덧붙이며 검색 원문·순위는 바꾸지 않습니다.
    """
    if input_namespace is None:
        raise RuntimeError("bind_team_backend로 팀 입력 함수를 먼저 연결하세요.")
    if not isinstance(state, dict):
        raise TypeError("게임 상태는 dict여야 합니다.")
    if not isinstance(player_input, str) or not player_input.strip():
        raise ValueError("플레이어 발언이 필요합니다.")
    target = canonical_npc_name(npc_name)
    # 이름이 질문에 이미 있어도 현재 상대라는 맥락을 유지합니다.
    # 실제 룰·시나리오 분류는 팀 I2의 규칙을 사용하므로 원문과 분류 결과를 함께 기록합니다.
    retrieval_query = f"대화 상대 {target}: {player_input}" if target else player_input
    started = time.perf_counter()
    input_stage = "retrieval"
    current_stage(input_stage, detail="팀 I2 검색 요청", player_input=player_input,
                  retrieval_query=retrieval_query, npc_name=target)
    try:
        raw = input_namespace["search_rag"](query=retrieval_query, state=deepcopy(state), k=3)
        searched = time.perf_counter()
        input_stage = "handoff"
        current_stage(input_stage, detail="팀 I3 전달 자료 구성", player_input=player_input,
            retrieval_query=retrieval_query, npc_name=target,
            route=raw.get("intent") if isinstance(raw, dict) else None,
            user_intent=raw.get("user_intent") if isinstance(raw, dict) else None,
            translated_queries=deepcopy(raw.get("translated_queries", [])) if isinstance(raw, dict) else None,
            sources_count={key: len(raw[key]) if isinstance(raw.get(key), list) else None
                for key in ("rule_results", "scenario_results")} if isinstance(raw, dict) else None,
            search_seconds=searched - started)
        original = input_namespace["handoff_builder"](search_result=raw, game_state=deepcopy(state))
    except Exception as error:
        # 검색 단계에서 실패해도 어떤 원문과 질의로 요청했는지 진단에 남깁니다.
        # 생성되지 않은 모델 메시지나 측정값은 만들어 채우지 않습니다.
        error.reply_to, error.retrieval_query = player_input, retrieval_query
        current_stage(input_stage, status="error", elapsed_seconds=time.perf_counter() - started,
                      error={"code": type(error).__name__, "message": str(error)})
        raise
    built = time.perf_counter()
    payload = read_handoff(original)
    payload["upstream_handoff"] = deepcopy(original)
    payload["player_input"] = player_input  # 검색용 문구를 실제 발언으로 저장하지 않습니다.
    payload["input_trace"] = {"original_player_input": player_input,
        "retrieval_query": retrieval_query, "intent": raw.get("intent"),
        "translated_queries": deepcopy(raw.get("translated_queries", []))}
    # 엔진의 추가 상태는 원본 handoff와 별도 필드로 둡니다.
    payload["engine_snapshot"] = deepcopy(state)
    payload["input_metrics"] = {"search_seconds": searched - started,
        "handoff_seconds": built - searched, "total_seconds": built - started}
    current_stage("handoff", detail="팀 전달 자료 준비 완료", status="completed",
                  input_metrics=deepcopy(payload["input_metrics"]),
                  input_trace=deepcopy(payload["input_trace"]))
    return payload


## 🟪 O3. 대화 기억·프롬프트

최근 대화와 생성 지침을 구성합니다. 자료의 시점·인물·물품 전달 방향을 구별합니다.


In [ ]:
# O3. 접속자별 대화와 프롬프트를 구성합니다.
# 아래 지침·최근 대화·팀 근거·확정 게임 상태가 실제 모델 입력이 됩니다.
_bae_load_output_support("context_helpers.py")

_dialogue_sessions = {}
_dialogue_lock = threading.RLock()


def create_memory_session():
    """이번 게임의 대화를 보관할 빈 공간을 만듭니다."""
    token = secrets.token_urlsafe(24)
    with _dialogue_lock:
        _dialogue_sessions[token] = {"histories": {}, "turn": 0}
    return token


def load_dialogue_history(session_id, npc):
    """이번 게임에서 현재 상대와 주고받은 말을 복사해서 가져옵니다."""
    with _dialogue_lock:
        return deepcopy(_dialogue_sessions.get(session_id, {}).get("histories", {}).get(npc, []))


def save_dialogue_history(session_id, npc, history):
    """누가 한 말인지 함께 저장하고 최근 기록 100개를 남깁니다."""
    with _dialogue_lock:
        session = _dialogue_sessions.setdefault(session_id, {"histories": {}, "turn": 0})
        session["histories"][npc] = normalize_history(history)[-100:]


def close_memory_session(session_id):
    """해당 게임의 대화 메모만 지웁니다. 다른 게임의 메모는 유지합니다."""
    with _dialogue_lock:
        _dialogue_sessions.pop(session_id, None)


# 원칙은 한 곳에서 공유합니다. 개별 인물·문장별 예외를 붙이지 않고,
# 원문 지식 / 확정 실행 / 실제 발언의 쓰임을 같은 구조로 구분합니다.
GROUNDING_SYSTEM = """자료를 사용하는 공통 원칙:
팀 원문은 사건·인물 관계의 근거, 엔진은 실행·위치·소지품·수치의 권한, 최근 대화는 누가 무슨 말을 했는지 잇는 맥락이다. 생성된 과거 대사를 새로운 설정이나 실행의 증명으로 삼지 않는다. 참고 문장은 대본이 아니며 인용 예시는 현재 발언이 아니다.
의미 보존 계약의 source_events는 원문 절과 그 절에 속한 시간이다. 사건마다 시점·기간·부정·조건·목격/전언의 주체를 유지한다. 기간을 특정 날짜로 좁히거나 다른 사건으로 옮기지 않는다. 자료에 없는 세부사항을 인물의 기억상실·새 경험·목격담으로 메우지 않고 확인된 범위에서 답한다.
speaker는 현재 화자, listener는 상대다. request_roles의 requester는 부탁한 쪽, performer는 수행하는 쪽이다. confirmed_transfers의 giver는 건넨 쪽, recipient는 받은 쪽이다. 이 역할에 맞춰 나·상대의 입장을 정하고, 부탁 수락에 대한 고마움과 물건을 받은 고마움을 구별한다. 제안·조건·과거 완료를 새로운 수락·지급·이동으로 바꾸지 않는다.
현재 장소·실제 허용 인물·공개된 지식과 확정 결과 안에서 표현한다. 소문·추측은 확정 사건으로 바꾸지 않고, 새로운 조사·행동·타인의 대사를 대신 만들어 진행하지 않는다."""

OUTPUT_SYSTEM = """당신은 한국어 TRPG의 현재 한 차례를 자연스러운 대사와 장면으로 작성한다.
""" + GROUNDING_SYSTEM + """
NPC는 지정된 인물의 1인칭과 말투로 직접 답한다. GM은 지금의 결과·관찰 또는 이번에 알게 된 정보를 이야기로 서술한다. 원문 보고서나 이전 장면의 반복으로 대신하지 않는다.
NPC의 직접 대사는 speech, GM의 서술은 narration으로 나누고 지정된 blocks JSON만 반환한다."""
NPC_CONVERSATION_SYSTEM = """당신은 지정된 현재 NPC로서 한국어로 한 차례 대화한다.
""" + GROUNDING_SYSTEM + """
상대의 최신 말에 먼저 한두 문장으로 직접 답한다. 일상적인 공감·농담·현재 감정 반응은 자연스럽게 표현하고, 묻지 않은 사건·의뢰 설명으로 되돌아가지 않는다. 상대의 알려지지 않은 이름·경험을 이미 아는 사실로 말하지 않는다.
성격·태도·말투는 연기 기준이다. 높임말과 종결형을 유지해 1~3문장으로 말한다. 서비스·가격은 현재 제공 목록으로 확인하며, 판매 목록을 새로운 서비스나 필수 준비물로 바꾸지 않는다. 진행 안내가 필요할 때만 allowed_guidance의 다음 대상을 짧게 연결한다.
현재 NPC의 speech 블록 JSON만 반환한다."""
NPC_CONVERSATION_FIRST_PASS = """인사·감사·감정·농담·잡담이나 대화 자체에 관한 말에는 현재 대화로 자연스럽게 답하고 mode=reply와 답변 blocks를 반환한다.
세계관·인물 관계나 이력·사건·장소·물품·규칙의 사실을 묻는 질문에는 mode=consult_reference, blocks=[]로 원문을 확인한다. 이전 대사만으로 사실 답변을 확정하거나 전체 목록이 완전하다고 가정하지 않는다.
JSON 형식은 {"mode":"reply 또는 consult_reference","blocks":[...]}이며 부가 설명을 붙이지 않는다."""
NPC_CONVERSATION_REFERENCE_PASS = """제공된 팀 원문에서 현재 질문에 필요한 사실을 확인해 현재 NPC의 말로 자연스럽게 답한다. 자료 문장을 대본처럼 읽지 않고 각 사건의 시간 범위와 소문·추측·확정의 차이를 유지한다. 확인되지 않은 세부사항은 꾸미지 않고 제공된 사실의 범위로 답한다. 자료 누락을 인물의 기억상실·새 경험·새 목격담으로 설명하지 않는다. 진행 안내는 현재 대화와 직접 연결될 때만 짧게 권한다.
이번에는 {"blocks":[...]} JSON만 반환한다. mode나 추가 자료 요청은 반환하지 않는다."""


def build_context(payload):
    """검색 원문·순서를 보존하고 표시용 자료 번호만 붙입니다."""
    sources = []
    for prefix, key in (("R", "rule_results"), ("S", "scenario_results")):
        for index, row in enumerate(payload["retrieval"].get(key, []), 1):
            meta = deepcopy(row.get("metadata") or {})
            sources.append({"label": prefix + str(index), "text": row["text"],
                "title": meta.get("title") or meta.get("card_id") or row.get("card_id") or key,
                "page": meta.get("page"), "score": row.get("score"), "metadata": meta})
    return "\n\n".join(row["text"] for row in sources), sources


def build_messages(payload, history=None, npc_name=None, engine_result=None):
    """팀 근거·현재 상태·최근 발언·이번 질문을 각자의 용도로 구분합니다.

    반환값은 Ollama에 보낼 메시지와 진단 표시용 출처 목록입니다.
    검색 문장을 재작성하거나 AI의 이전 대사에서 인물 설정을 추출하지 않습니다.
    """
    speaker = output_speaker(payload, npc_name)
    if npc_name is not None and payload.get("npc_name") != speaker:
        payload = _with_speaker_whitelist(payload, npc_name)
    _, sources = build_context(payload)
    contract = _utterance_task(payload)
    npc_mode = contract["mode"] == "npc_dialogue"
    if (contract["phase"] == "action_result" and contract["mode"] != "rule_explanation"
            and isinstance(engine_result, dict) and engine_result.get("confirmed") is True):
        return _build_current_action_messages(payload, speaker, contract, engine_result, history)
    state = payload.get("engine_snapshot") or payload.get("model_state") or payload["game_state"]
    if not isinstance(state, dict):
        raise ValueError("현재 모델용 상태는 dict여야 합니다.")
    # 검색 조건용 내부 vars·flags는 모델의 관찰 사실이 아닙니다.
    public_keys = {"scene", "cls", "current_room", "inventory", "gold_cp", "money", "hero", "combat",
        "플레이어 이름", "현재 장소", "플레이어 소지금", "플레이어 장착 장비", "플레이어가 보유한 소지품"}
    public_state = {key: value for key, value in state.items() if key in public_keys}
    public_state = _engine_data(public_state)
    snapshot = payload.get("player_snapshot")
    if isinstance(snapshot, dict):
        # 하나의 현재 플레이어 상태로 합쳐 가방·금액·HP를 중복 제시하지 않습니다.
        player = public_state.pop("hero", {})
        player = player if isinstance(player, dict) else {}
        for key in ("inventory", "gold_cp", "money"):
            if key in public_state:
                player[key] = public_state.pop(key)
        player.update(_engine_data(snapshot))
        public_state["player"] = player
    if npc_mode:
        public_state.pop("플레이어 이름", None)
        for key in ("player", "hero"):
            if isinstance(public_state.get(key), dict):
                public_state[key].pop("name", None)
                public_state[key].pop("id", None)
    # 참조 ID는 모델 입력의 사본에만 붙입니다. 원본 I3 문장·순서·출처는 유지합니다.
    referenced_sources = {}
    for prefix, key, group in (("R", "rule_results", "rules"), ("S", "scenario_results", "scenarios")):
        referenced_sources[group] = [{**deepcopy(row), "source_id": prefix + str(index)}
            for index, row in enumerate(payload["retrieval"].get(key, []), 1)
            if prefix + str(index) in contract["context_source_ids"]]
    data = {"현재 NPC": speaker or None,
        "팀 RAG 검색 결과": referenced_sources,
        "현재 엔진 상태": public_state,
        "읽기 전용 상호작용 계약": _current_interaction_contract(payload)}
    data.update(_speaker_context(payload))
    if npc_mode:
        data["dialogue_state"] = _npc_view(data["dialogue_state"])
    # 실행 ID 선택은 입력·추천 단계의 일입니다. 본문에는 공개 장면·진행 맥락을 줍니다.
    for key in ("scene_context", "story_progression", "progression_options", "participant_roles",
                "action_subject", "executed_action_ids", "engine_delta", "item_handovers"):
        if key in payload:
            value = _engine_data(payload[key])
            data[key] = _npc_view(value) if npc_mode else value
    if "dialogue_brief" in payload:
        data["dialogue_brief"] = deepcopy(payload["dialogue_brief"])
    data["진행 안내"] = _story_context(payload)
    if "instructions" in payload:
        data["팀 입력 지침"] = deepcopy(payload["instructions"])
    if engine_result is not None:
        # 대화 중에는 마지막 행동을 새로 실행한 것처럼 표현하지 않습니다.
        phase = (payload.get("dialogue_state") or {}).get("phase")
        key = "직전에 확정된 행동 결과" if phase == "continue" else "이번에 확정된 행동 결과"
        confirmed = _engine_data(validate_engine_result(engine_result))
        if payload.get("engine_snapshot"):
            confirmed.pop("state_after", None)  # 현재 상태는 위에 한 번 제시합니다.
        data[key] = _npc_view(confirmed) if npc_mode else confirmed
    if npc_mode:
        # NPC 대사 작성에는 검색 점수·전체 화자 목록·행동 라우팅 구조가 필요하지
        # 않습니다. 원문과 확정 사건은 유지하고 모델에게 보여 주는 사본만 줄입니다.
        # 원본 I3·엔진·전체 진단 기록은 여기서 수정하지 않습니다.
        state_for_dialogue = {key: deepcopy(public_state[key]) for key in
            ("scene", "current_room", "현재 장소", "inventory", "gold_cp", "money",
             "플레이어 소지금", "플레이어가 보유한 소지품") if key in public_state}
        if isinstance(public_state.get("player"), dict):
            state_for_dialogue["player"] = {key: deepcopy(public_state["player"][key]) for key in
                ("inventory", "gold_cp", "money") if key in public_state["player"]}
        data = {"이번 화자": deepcopy(contract["speaker"]),
            "이번 대화의 역할": _npc_turn_roles(payload, contract),
            "화자의 인적 사항 원문 번호": contract["speaker_source_ids"],
            "말투": _speech_style(contract["speaker"]["speaker_id"]),
            "팀 RAG 검색 결과": {group: [{"source_id": row["source_id"], "text": row["text"]}
                for row in rows] for group, rows in referenced_sources.items()},
            "현재 엔진 상태": state_for_dialogue,
            "현재 공개 상호작용": _npc_view(_engine_data(payload.get("interaction_context") or {})),
            "읽기 전용 상호작용 계약": _current_interaction_contract(payload),
            "진행 안내": _story_context(payload),
            **{key: value for key, value in data.items() if key in
                ("이번에 확정된 행동 결과", "직전에 확정된 행동 결과")}}
        affordances = _npc_dialogue_affordances(payload)
        if affordances is not None:
            data["현재 장소의 이용 조건"] = affordances
    actual_sources = [row for row in sources if row["label"] in contract["context_source_ids"]]
    data["의미 보존 계약"] = _meaning_preservation_context(payload, contract, actual_sources, engine_result)
    start = {"role": "system", "content": NPC_CONVERSATION_SYSTEM if npc_mode else OUTPUT_SYSTEM}
    context = {"role": "system", "content": "이번 응답의 참고 자료입니다. 등장인물의 발언이 아닙니다.\n" + _json_text(data)}
    turn = {"role": "system", "content": "이번 턴 계약\n" + contract["task"] + "\n" + _json_text(contract)}
    current = {"role": "user", "content": payload["player_input"]}
    recent = _recent_messages(history, speaker_id=contract["speaker"]["speaker_id"])
    if contract["mode"] == "scene_dialogue":
        recent = []  # 새 장면의 사실을 이전 생성 문장에서 가져오지 않습니다.
    messages = [start, *recent, context, turn, current]
    # 원본 검색 결과·현재 질문을 유지하고 오래된 대화만 줄입니다.
    while recent and _prompt_characters(messages) > MAX_INPUT_CHARACTERS:
        recent.pop(0)
        messages = [start, *recent, context, turn, current]
    # 검색 전체 원본은 payload에 남고, 이 목록은 실제 모델에 제시한 자료입니다.
    return messages, actual_sources


def build_npc_conversation_messages(payload, history, speaker, include_reference=False, engine_result=None):
    """대화를 먼저 이어가고 꼭 필요한 사실만 원문에서 확인하도록 구성합니다.

    잡담에도 현재 역할·장소·짧은 엔진 상태를 유지합니다. 사실 확인이 필요한
    요청에만 이미 받은 팀 I3 원문을 더합니다. 원본 검색·상태·이력은 바꾸지
    않으며 대화에서 인물 설정을 추출하거나 별도 사실 캐시를 만들지 않습니다.
    """
    current_speaker = output_speaker(payload, speaker)
    local = payload if payload.get("npc_name") == current_speaker else _with_speaker_whitelist(payload, current_speaker)
    contract = _utterance_task(local)
    if contract["mode"] != "npc_dialogue":
        raise ValueError("자유 대화 구성에는 현재 NPC 대화 모드가 필요합니다.")
    identifier = contract["speaker"]["speaker_id"]
    style = _speech_style(identifier)
    data = {"현재 화자": deepcopy(contract["speaker"]), "말투": style,
        "이번 대화의 역할": _npc_turn_roles(local, contract),
        "읽기 전용 상호작용 계약": _current_interaction_contract(local),
        "현재 엔진 상태": _npc_view(_current_public_state(local)),
        "현재 장소": _npc_current_scene(local),
        "이번 입력": {"phase": contract["phase"], "input_kind": contract["input_kind"]},
        "출력 계약": deepcopy(contract["output"])}
    # 질문이 잡담으로 분류돼도 가방과 현재 서비스 조건이 사라지지 않습니다.
    # 아직 발견하지 못한 조건은 연결부가 이 자료에 넣지 않습니다.
    affordances = _npc_dialogue_affordances(local)
    if affordances is not None:
        data["현재 장소의 이용 조건"] = affordances
    actual_sources = []
    fact_scope = local.get("fact_scope")
    if fact_scope not in {"world", "state", "services", "requirements", "conversation"}:
        fact_scope = "services" if local.get("needs_services") is True else None
    data["사실 확인 범위"] = fact_scope
    engine_scope = fact_scope in {"state", "services", "requirements"}
    data["진행 안내"] = _story_context(local)
    if include_reference and not engine_scope:
        _, actual_sources = build_context(local)
        full_references = {"rules": [], "scenarios": []}
        for source in actual_sources:
            group = "rules" if source["label"].startswith("R") else "scenarios"
            full_references[group].append({"source_id": source["label"], "text": source["text"]})
        data["팀 RAG 검색 결과"] = full_references
        data["현재 공개 상호작용"] = _npc_view(_engine_data(local.get("interaction_context") or {}))
        data["진행 안내"] = _story_context(local)
        # 이전 행동의 전체 장면 문장은 재사용하지 않고 실제 전달 영수증만 제공합니다.
        if isinstance(engine_result, dict):
            data["확인된 물품 전달"] = _npc_view(_engine_data(engine_result.get("item_handovers") or []))
    data["의미 보존 계약"] = _meaning_preservation_context(local, contract, actual_sources, engine_result)
    instruction = NPC_CONVERSATION_REFERENCE_PASS if include_reference else NPC_CONVERSATION_FIRST_PASS
    if fact_scope == "services":
        # 가격·이용 가능 여부를 묻는 경우 공동 판매 카드로 새 서비스를 만들지
        # 않게 현재 장소의 엔진 목록만 사용합니다. 원본 retrieval은 그대로 남습니다.
        instruction = ("지금 제공 가능한 물품 판매와 서비스를 구별해 이용객에게 답한다. "
            "services_complete=true이면 가능한 서비스와 가격은 그 목록으로 확인한다. "
            "제공하지 않는 것은 가게에서 현재 준비하지 않은 것으로 자연스럽게 설명하고, 미확인 조건은 확인이 필요하다고 말한다. "
            "현재 소지품 수량을 상품의 단위나 판매 수량으로 사용하지 않는다. "
            + ('{"blocks":[...]} JSON만 반환한다.' if include_reference else
               '{"mode":"reply","blocks":[...]} JSON만 반환한다.'))
    elif fact_scope in {"requirements", "state"}:
        instruction = ("현재 엔진 상태와 공개된 이용 조건으로만 직접 답한다. 현재 소지품과 장비의 필요 조건은 별개다. "
            "equipment_checks가 비어 있으면 실제 필요한 장비 조건은 아직 확인되지 않은 상태다. "
            "그때는 확인된 소지품과 이용 가능한 도움을 말하고 현장 조건을 먼저 살펴보도록 권한다. "
            "판매 물품을 필수 준비물로 추정하지 않는다. "
            + ('{"blocks":[...]} JSON만 반환한다.' if include_reference else
               '{"mode":"reply","blocks":[...]} JSON만 반환한다.'))
    start = {"role": "system", "content": NPC_CONVERSATION_SYSTEM + "\n" + instruction}
    context = {"role": "system", "content": "현재 대화의 참고 자료입니다. 대화 이력은 실제 발언이며 사실의 증명은 아닙니다.\n" + _json_text(data)}
    # 참고 자료를 먼저 주고 실제 대화를 이어 붙여 가장 최근 발언을 마지막에 둡니다.
    recent = _plain_npc_conversation(history, identifier)
    current = {"role": "user", "content": local["player_input"]}
    messages = [start, context, *recent, current]
    while recent and _prompt_characters(messages) > MAX_INPUT_CHARACTERS:
        recent.pop(0)
        messages = [start, context, *recent, current]
    return messages, actual_sources


## 🟪 O4. Qwen 호출·답변 작성

모델로 자연스러운 대사를 작성하고, 코드가 확정한 이동·처리 결과는 보조 코드에서 표시합니다.


In [ ]:
# O4. Ollama에 생성 요청을 보내고 NPC 대사와 행동 결과를 만듭니다.
# 스트리밍·형식 검사·오류 정리는 보조 함수가 담당합니다.
_bae_load_output_support("response_validation.py")
_bae_load_output_support("generation_helpers.py")

def load_output_model(pull=False):
    """설치된 Ollama 모델을 확인합니다. pull=True일 때만 내려받습니다."""
    global output_model
    from ollama_backend import OllamaBackend
    started = time.perf_counter()
    current_stage("lock_wait", detail="모델 연결 잠금 대기", model=MODEL_NAME)
    with generation_lock:
        if output_model is None:
            backend = OllamaBackend(MODEL_NAME, OLLAMA_BASE_URL,
                keep_alive=OLLAMA_KEEP_ALIVE, num_ctx=OLLAMA_NUM_CTX)
            current_stage("model_loading", detail="설치된 모델 정보 확인", model=MODEL_NAME, pull=pull)
            try:
                backend.prepare(pull=pull)
            except BaseException as error:
                current_stage("model_loading", status="error", elapsed_seconds=time.perf_counter() - started,
                              error={"code": type(error).__name__, "message": str(error)})
                raise
            output_model = backend
    current_stage("model_ready", detail="출력 모델 연결 확인 완료", backend=diagnostics_metadata(),
                  elapsed_seconds=time.perf_counter() - started)
    return output_model


def run_generation(messages, max_new_tokens=MAX_NEW_TOKENS, sample=None, thinking=False,
                   temperature=None, top_p=None, top_k=None, on_text=None, cancel_event=None,
                   response_format=None):
    """한 요청을 실행하고 실제 토큰·시간·적용 옵션을 함께 반환합니다.

    sample이 없으면 공통 출력 설정을 따릅니다. 실패 시에도 시도한 옵션을
    예외에 붙여 다음 진단 단계가 추정값을 기록하지 않도록 합니다.
    """
    # 블록이 완성되기 전에도 UI 종료·추천 취소를 같은 경로에서 확인합니다.
    inherited_events = _output_stream_cancel_events.get()
    check_cancel = lambda: _raise_if_cancelled(cancel_event, inherited_events=inherited_events)
    check_cancel()
    if output_model is None:
        raise RuntimeError("출력 모델을 먼저 준비하세요.")
    if _prompt_characters(messages) > MAX_INPUT_CHARACTERS:
        raise ValueError("전달 자료가 너무 깁니다. 팀 검색 개수나 입력 분량을 확인하세요.")
    if type(max_new_tokens) is not int or not 0 < max_new_tokens < OLLAMA_NUM_CTX:
        raise ValueError("생성 한도는 Ollama 문맥 한도보다 작은 양의 정수여야 합니다.")
    format_options = {"response_format": response_format} if response_format is not None else {}
    sampled = DO_SAMPLE if sample is None else sample
    options = {"num_ctx": OLLAMA_NUM_CTX, "num_predict": max_new_tokens,
        "sample": sampled, "temperature": (TEMPERATURE if temperature is None else temperature) if sampled else 0,
        "top_p": (TOP_P if top_p is None else top_p) if sampled else None,
        "top_k": (TOP_K if top_k is None else top_k) if sampled else None,
        "thinking": thinking, "structured_response": response_format is not None}
    # 적용할 설정과 실제 요청을 먼저 남겨 응답이 오지 않은 호출도 확인합니다.
    started = time.perf_counter()
    current_stage("model_wait", backend=diagnostics_metadata(), request_options=deepcopy(options),
                  request_messages=deepcopy(messages))
    first_text = False
    active_stage = "model_wait"
    def traced_text(text):
        nonlocal first_text
        if not first_text:
            first_text = True
            current_stage("generating", detail="첫 본문 도착",
                          first_text_seconds=time.perf_counter() - request_started)
        if on_text is not None:
            return on_text(text)
    try:
        with _generation_slot(cancel_event):
            active_stage = "generating"
            request_started = time.perf_counter()
            current_stage("generating", detail="Ollama 본문 요청", backend=diagnostics_metadata(),
                          request_options=deepcopy(options))
            result = output_model.generate(messages, max_new_tokens=max_new_tokens,
                sample=sampled, thinking=thinking,
                temperature=TEMPERATURE if temperature is None else temperature,
                top_p=TOP_P if top_p is None else top_p, top_k=TOP_K if top_k is None else top_k,
                on_text=traced_text, check_cancel=check_cancel, **format_options)
            check_cancel()
    except BaseException as error:
        error.request_options = deepcopy(options)
        current_stage(active_stage, status="error", elapsed_seconds=time.perf_counter() - started,
                      error={"code": type(error).__name__, "message": str(error)})
        raise
    result["generation_limit"] = max_new_tokens
    result["request_options"] = options
    current_stage("validation", detail="모델 응답 수신 완료", backend=diagnostics_metadata(),
                  request_options=deepcopy(options),
                  model_metrics={key: result[key] for key in ("seconds", "load_seconds", "prompt_seconds",
                      "decode_seconds", "input_tokens", "output_tokens", "done_reason") if key in result})
    return result


def generate_from_handoff(value, history=None, npc_name=None, engine_result=None):
    """원본 I3 JSON을 직접 받아 검색 결과와 최근 대화로 본문을 생성합니다."""
    payload = _with_speaker_whitelist(read_handoff(value), npc_name)
    speaker = output_speaker(payload, npc_name)
    engine_result = engine_result if engine_result is not None else payload.get("engine_result")
    if isinstance(engine_result, dict) and (payload.get("dialogue_state") or {}).get("phase") == "action_result":
        payload["engine_result"] = validate_engine_result(engine_result)
    payload = _with_action_speaker_scope(payload, engine_result)
    task = _utterance_task(payload)
    if task["mode"] == "npc_dialogue" and task["phase"] == "continue" and task["input_kind"] == "speech":
        # 이 경로는 별도 대화 프롬프트를 만듭니다. 사용하지 않을 일반 프롬프트는
        # 구성하지 않되, 기존 현재 상태·확정 사건의 입력 형식 검사는 유지합니다.
        state = payload.get("engine_snapshot") or payload.get("model_state") or payload["game_state"]
        if not isinstance(state, dict):
            raise ValueError("현재 모델용 상태는 dict여야 합니다.")
        if engine_result is not None and (not isinstance(engine_result, dict)
                or engine_result.get("confirmed") is not True):
            raise ValueError("엔진이 확정한 행동 결과만 전달하세요.")
        return _generate_npc_conversation(payload, history, speaker, engine_result=engine_result)
    messages, sources = build_messages(payload, history, speaker, engine_result)
    prepared = _prepared_opening(payload, speaker, sources, engine_result=engine_result)
    if prepared is not None:
        return prepared
    outcome = _prepared_confirmed_interaction(payload, speaker, sources, engine_result=engine_result)
    if outcome is not None:
        return outcome
    return _generate_main_result(messages, payload, speaker, sources, engine_result)


def generate_action_result(payload, *, handoff=None, engine_result=None):
    """팀 I3 근거와 별도 확정 사건으로 행동 뒤의 표현을 생성합니다.

    handoff 또는 payload.input_handoff에 이미 검색한 팀 JSON을 전달합니다.
    출력은 재검색하지 않습니다. handoff 없는 호출은 엔진 결과만으로
    처리하며 public_narratives의 직접 카드 본문은 생성 입력에 넣지 않습니다.
    """
    if not isinstance(payload, dict):
        raise TypeError("행동 결과는 dict여야 합니다.")
    caller = deepcopy(payload)
    confirmed = validate_engine_result(engine_result if engine_result is not None else caller.get("engine_result"))
    original = handoff if handoff is not None else caller.get("input_handoff")
    if original is None and ("retrieval" in caller or "retrieved_context" in caller):
        original = caller
    if original is not None:
        prepared = read_handoff(original)
        evidence_status = "team_handoff"
    else:
        state = caller.get("engine_snapshot") or confirmed.get("state_after") or caller.get("game_state") or {}
        prepared = read_handoff({"player_input": caller.get("player_input"), "game_state": _engine_data(state),
            "retrieval": {"rule_results": [], "scenario_results": [], "memory_results": []}})
        evidence_status = "engine_only_no_input_handoff"
    # 실행 시점의 역할·수치만 별도로 연결합니다. 지식·카드 본문은 I3에서만 받습니다.
    for key in ("npc_name", "actor_context", "speaker_id", "allowed_speakers", "dialogue_state",
                "response_mode", "action_kind", "movement_scene", "interaction_context",
                "story_progression", "confirmed_offer", "needs_information", "reference_required", "turn_intent",
                "scene_context", "player_snapshot", "engine_snapshot", "model_state", "available_actions", "dialogue_affordances",
                "progression_options", "participant_roles", "action_subject", "executed_action_ids",
                "engine_delta", "item_handovers"):
        if key in caller:
            prepared[key] = _engine_data(caller[key])
    phase = deepcopy(prepared.get("dialogue_state") or {})
    if not isinstance(phase, dict):
        raise ValueError("dialogue_state는 객체여야 합니다.")
    phase.setdefault("phase", "action_result")
    prepared["dialogue_state"] = phase
    prepared["engine_result"] = confirmed
    prepared = _with_speaker_whitelist(prepared)
    prepared = _with_action_speaker_scope(prepared, confirmed)
    speaker = output_speaker(prepared)
    if not isinstance(prepared.get("player_input"), str) or not prepared["player_input"].strip():
        raise ValueError("실행한 행동의 문구가 필요합니다.")
    messages, sources = build_messages(prepared, caller.get("recent_conversation"), speaker, confirmed)
    opening = _prepared_opening(prepared, speaker, sources, engine_result=confirmed)
    if opening is not None:
        opening["evidence_status"] = evidence_status
        return opening
    outcome = _prepared_confirmed_interaction(prepared, speaker, sources, engine_result=confirmed)
    if outcome is not None:
        outcome["evidence_status"] = evidence_status
        return outcome
    result = _generate_main_result(messages, prepared, speaker, sources, confirmed)
    result["evidence_status"] = evidence_status
    if evidence_status != "team_handoff":
        result["warnings"].append("팀 입력 근거가 없어 엔진 확정 결과만 사용했습니다.")
    return result


## 🟪 O5. 다음 선택지

현재 장소·소개받은 장소·실행 조건에 맞는 다음 선택지를 제시합니다.


In [ ]:
# O5. 방금 대화와 현재 가능한 행동으로 후속 선택지를 추천합니다.
# 추천은 문장과 엔진 행동 ID를 연결하며 게임 상태를 변경하지 않습니다.
def generate_followup_question(payload, reply, history=None, cancel_event=None):
    """방금 대화와 실제 가능한 행동으로 다음 선택지를 만듭니다.

    대화 추천은 입력 문장이고, 행동 추천은 엔진 ID에 연결됩니다.
    추천 생성은 세계 상태나 소지품을 바꾸지 않습니다.
    """
    _raise_if_cancelled(cancel_event)
    count = max(1, min(3, int(payload.get("suggestion_count", SUGGESTION_COUNT))))
    actions = deepcopy(payload.get("available_actions", []))
    data = {"현재 NPC": output_speaker(payload), "플레이어 원문": payload.get("player_input"),
        "방금 본문": reply, "최근 대화": _recent_messages(history)[-6:], "실제 가능한 행동": actions}
    messages = [{"role": "system", "content":
        f'이번 대화에 이어 플레이어가 다음에 할 말이나 행동을 {count}개 추천한다.\n'
        '현재 질문의 목적을 이어가는 짧은 선택지를 만들고, 플레이어가 말하지 않은 목격·경험을 지어내지 않는다.\n'
        '대화는 플레이어가 할 말이고 행동·이동은 제공된 현재 행동 ID와 그 이름을 그대로 사용한다.\n'
        'JSON 계약: {"options":[{"kind":"dialogue|action|move","text":"플레이어의 말 또는 행동","action_id":"연결할 때의 실제 ID"}]}.'},
        {"role": "user", "content": _json_text(data)}]
    generated = run_generation(messages, max_new_tokens=160, sample=False, cancel_event=cancel_event)
    _raise_if_cancelled(cancel_event)
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", generated["text"].strip()).strip()
    try:
        value = json.loads(text)
    except ValueError:
        value = {}
    allowed = {row["id"]: row for row in actions if isinstance(row, dict) and isinstance(row.get("id"), str)
               and isinstance(row.get("label"), str) and row["label"].strip()}
    options = []
    from response_guard import inspect_response
    rows = value.get("options", []) if isinstance(value, dict) else []
    for row in rows if isinstance(rows, list) else []:
        if not isinstance(row, dict) or row.get("kind") not in {"dialogue", "action", "move"}:
            continue
        label = row.get("text")
        if (not isinstance(label, str) or not 1 <= len(label.strip()) <= 140
                or inspect_response(label)["hard_issues"]):
            continue
        identifier = row.get("action_id")
        if identifier and (not isinstance(identifier, str) or identifier not in allowed):
            continue
        if row["kind"] != "dialogue" and not identifier:
            continue  # 실행할 수 있는 행동만 추천합니다.
        if row["kind"] == "dialogue" and identifier:
            continue  # 대화 문장을 누른 뒤 다른 행동이 실행되지 않게 합니다.
        option = {"kind": row["kind"], "text": label.strip()}
        if identifier:
            option["action_id"] = identifier
            # 화면의 행동 이름과 클릭 후 실행되는 행동이 서로 달라지지 않게 합니다.
            option["text"] = allowed[identifier]["label"]
            option["kind"] = "move" if identifier.startswith("move:") else "action"
        if option not in options:
            options.append(option)
        if len(options) >= count:
            break
    # 선택지가 없는 원인을 구분해 진단에 남깁니다. 임의의 고정 답변으로 채우지 않습니다.
    truncated = generated.get("done_reason") == "length" or generated.get("output_tokens", 0) >= generated.get("generation_limit", 160)
    if options:
        reason = "generated"
    elif truncated:
        reason = "generation_limit"
    elif not isinstance(value, dict) or not isinstance(value.get("options"), list):
        reason = "invalid_json"
    else:
        reason = "no_valid_options"
    return {"text": options[0]["text"] if options else "", "suggestions": [row["text"] for row in options],
        "recommendation_options": options, "seconds": generated["seconds"],
        "input_tokens": generated["input_tokens"], "output_tokens": generated["output_tokens"], "reason": reason}


## 🟪 O6. 게임 화면 열기

Gradio 게임 화면을 엽니다. 실제 모델·GPU 정보와 전투 결과 연출을 표시합니다.


In [ ]:
# O6. 위에서 정의한 출력 함수와 팀 함수를 연결해 Gradio 게임을 엽니다.
# 화면 구성·파일 버전 확인은 읽을 수 있는 보조 코드에 모았습니다.
_bae_load_output_support("gui_bridge.py")

bae_demo = _bae_connect_and_open(globals())


## 🟪 실행

팀 공통·룰·시나리오·입력 준비 후 O1~O6을 실행합니다. 보조 코드는 `BAE/src`에 공개되어 있습니다. 파일 선택이나 중복 모델 설치는 없습니다.


# 🟩 외부주소로 열기(임시 배포) Colab + ngrok

In [ ]:
# [ngrok] 팀 Gradio 화면(O6) 외부 주소 열기 — O6 셀을 실행해서 Gradio가 열린 뒤에 실행하세요
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
from pyngrok import ngrok
try:
    from google.colab import userdata
    gr_ngrok_token = userdata.get("NGROK_TOKEN")
except Exception:
    gr_ngrok_token = None
if not gr_ngrok_token:
    raise RuntimeError("NGROK_TOKEN이 없어요. 🔑 보안 비밀에 NGROK_TOKEN을 추가하고 '노트북 액세스'를 켠 뒤 다시 실행하세요.")
ngrok.kill()
ngrok.set_auth_token(gr_ngrok_token)
gr_port = getattr(bae_demo, "server_port", None) or 7860      # O6이 연 Gradio의 포트 (못 찾으면 기본 7860)
gr_url = ngrok.connect(gr_port, "http").public_url
print("Gradio 주소:", gr_url, "(포트", gr_port, ")")
print("※ ngrok 무료 플랜은 처음 접속할 때 안내 화면(Visit Site 버튼)이 뜰 수 있어요. 127.0.0.1 주소는 열지 마세요.")

Gradio 주소: https://lining-flattery-penny.ngrok-free.dev (포트 7860 )
※ ngrok 무료 플랜은 처음 접속할 때 안내 화면(Visit Site 버튼)이 뜰 수 있어요. 127.0.0.1 주소는 열지 마세요.
